In [1]:
!pip install trl transformers accelerate peft datasets bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.1 MB/s eta 0:00:00


In [2]:
!pip install -U torchao --break-system-packages -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 7.6 MB/s eta 0:00:00


In [3]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

KD_BASE_PATH = "/kaggle/input/datasets/mythreyeehari20/kd-abdomen-weights/kd_merged_abdomen"
QLORA_ADAPTER_PATH = "/kaggle/input/datasets/mythreyeehari20/abdomen-kd-qlora-best-r32-alpha64/qlora_best_r32_alpha64"

tokenizer = AutoTokenizer.from_pretrained(KD_BASE_PATH, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base_for_merge = AutoModelForCausalLM.from_pretrained(
    KD_BASE_PATH, torch_dtype=torch.bfloat16, device_map={"": 0}, trust_remote_code=True,
)
merge_model = PeftModel.from_pretrained(base_for_merge, QLORA_ADAPTER_PATH)
merged_model = merge_model.merge_and_unload()

FINAL_MERGED_DIR = "/kaggle/working/kd_qlora_merged_final"
merged_model.save_pretrained(FINAL_MERGED_DIR)
tokenizer.save_pretrained(FINAL_MERGED_DIR)
print(f"Merged KD+QLoRA model saved to: {FINAL_MERGED_DIR}")

del base_for_merge, merge_model
gc.collect()
torch.cuda.empty_cache()

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged KD+QLoRA model saved to: /kaggle/working/kd_qlora_merged_final


In [4]:
EVAL_BATCH_SIZE = 2
EVAL_STEPS = 50

def extract_gold(record):
    """Pull gold labels back out of the assistant turn of an SFT-formatted record."""
    assistant_turn = next(m["content"] for m in record["messages"] if m["role"] == "assistant")
    return json.loads(assistant_turn)

def generate_predictions(model, tokenizer, records, few_shot_pool=None, n_shot=0,
                          n=None, batch_size=EVAL_BATCH_SIZE):
    import transformers
    transformers.logging.set_verbosity_error()
    model.eval()

    subset = records[:n] if n else records
    results = []

    for start in range(0, len(subset), batch_size):
        batch_records = subset[start:start + batch_size]
        texts = [
            tokenizer.apply_chat_template(
                build_messages(r, few_shot_pool, n_shot),
                tokenize=False, add_generation_prompt=True
            )
            for r in batch_records
        ]
        inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        prompt_len = inputs["input_ids"].shape[-1]
        for i, record in enumerate(batch_records):
            generated = output_ids[i][prompt_len:]
            raw = tokenizer.decode(generated, skip_special_tokens=True).strip()
            parsed = parse_output(raw)
            gold = extract_gold(record)

            results.append({
                "report_id": record.get("report_id", start + i),  # synthetic id — not present in these records
                "gold_sentences": gold["incidental_sentences"],
                "pred_sentences": parsed.get("incidental_sentences", []) if parsed else None,
                "parse_failed": parsed is None,
            })

    return results

In [5]:
def extract_report_text(record):
    """Pull the raw report free-text back out of an already-formatted SFT record."""
    user_turn = next(m["content"] for m in record["messages"] if m["role"] == "user")
    return user_turn.removeprefix("Report:\n")

def build_messages(record, few_shot_pool=None, n_shot=0):
    """mod #3: ACR-augmented system prompt is built per-record instead of a
    single flat SYSTEM_PROMPT."""
    report_text = extract_report_text(record)
    messages = [{"role": "system", "content": build_system_prompt(report_text)}]

    if few_shot_pool and n_shot > 0:
        for ex in few_shot_pool[:n_shot]:
            ex_text = extract_report_text(ex) if "messages" in ex else ex["free_text"]
            messages.append({"role": "user", "content": f"Report:\n{ex_text}"})
            messages.append({
                "role": "assistant",
                "content": json.dumps({
                    "contains_IF": ex["gold"]["contains_IF"],
                    "incidental_sentences": ex["gold"]["incidental_sentences"],
                }),
            })

    messages.append({"role": "user", "content": f"Report:\n{report_text}"})
    return messages

In [6]:
def parse_output(raw_text):
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        pass
    match = re.search(r"\{.*\}", raw_text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return None

In [7]:
from difflib import SequenceMatcher

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def fuzzy_match_sets(gold_set, pred_set, threshold=0.85):
    """Greedy one-to-one fuzzy matching between gold and predicted sentences."""
    gold_list = list(gold_set)
    pred_list = list(pred_set)
    matched_gold, matched_pred = set(), set()
    pairs = []
    for gi, g in enumerate(gold_list):
        for pi, p in enumerate(pred_list):
            sim = similarity(g, p)
            if sim >= threshold:
                pairs.append((sim, gi, pi))
    pairs.sort(key=lambda x: -x[0])
    for sim, gi, pi in pairs:
        if gi in matched_gold or pi in matched_pred:
            continue
        matched_gold.add(gi)
        matched_pred.add(pi)
    tp = len(matched_gold)
    fp = len(pred_list) - len(matched_pred)
    fn = len(gold_list) - len(matched_gold)
    return tp, fp, fn


def run_evaluation_fuzzy(results, label="", threshold=0.85):
    parse_failures = sum(r["parse_failed"] for r in results)
    valid = [r for r in results if not r["parse_failed"]]
    total_tp = total_fp = total_fn = 0
    neg_scores, pos_scores = [], []
    for r in valid:
        gold_set = set(s.strip().lower() for s in r["gold_sentences"])
        pred_set = set(s.strip().lower() for s in (r["pred_sentences"] or []))
        if len(gold_set) == 0:
            neg_scores.append(1.0 if len(pred_set) == 0 else 0.0)
        else:
            tp, fp, fn = fuzzy_match_sets(gold_set, pred_set, threshold=threshold)
            total_tp += tp; total_fp += fp; total_fn += fn
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
            pos_scores.append(f1)
    avg_neg = sum(neg_scores) / len(neg_scores) if neg_scores else 0.0
    avg_pos = sum(pos_scores) / len(pos_scores) if pos_scores else 0.0
    n_neg, n_pos = len(neg_scores), len(pos_scores)
    if n_neg > 0 and n_pos > 0:
        macro_f1 = (avg_neg + avg_pos) / 2
    elif n_neg > 0:
        macro_f1 = avg_neg
    elif n_pos > 0:
        macro_f1 = avg_pos
    else:
        macro_f1 = 0.0
    weighted_f1 = (n_neg * avg_neg + n_pos * avg_pos) / (n_neg + n_pos) if (n_neg + n_pos) > 0 else 0.0
    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    micro_f1 = (2 * micro_precision * micro_recall / (micro_precision + micro_recall)
                if (micro_precision + micro_recall) > 0 else 0.0)
    print(f"\n{'='*50}\n=== {label} (fuzzy threshold={threshold}) ===\n{'='*50}")
    print(f"Parse failures:      {parse_failures}/{len(results)}")
    print(f"Negative-report acc: {avg_neg:.4f}  (n={n_neg})")
    print(f"Positive-report F1:  {avg_pos:.4f}  (n={n_pos})")
    print(f"Sentence Macro F1:   {macro_f1:.4f}")
    print(f"Sentence Weighted:   {weighted_f1:.4f}")
    print(f"Sentence Micro F1:   {micro_f1:.4f}")
    return {"macro_f1": macro_f1, "weighted_f1": weighted_f1, "micro_f1": micro_f1,
            "parse_failures": parse_failures, "n_neg": n_neg, "n_pos": n_pos,
            "avg_neg": avg_neg, "avg_pos": avg_pos}


In [8]:
import json

# --- Load your pre-split abdomen train/test JSONL files ---
ABDOMEN_TRAIN_PATH = "/kaggle/input/datasets/mythreyee1006/train-dataset-abdomen-new/train_records_abdomenCT.jsonl"  # adjust path
ABDOMEN_TEST_PATH = "/kaggle/input/datasets/mythreyee1006/test-dataset-abdomen-final/test_records_abdomenCT.jsonl"    # adjust path



def load_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records


train_records_raw = load_jsonl(ABDOMEN_TRAIN_PATH)
test_records_raw = load_jsonl(ABDOMEN_TEST_PATH)
print(f"Train records loaded: {len(train_records_raw)}")
print(f"Test records loaded: {len(test_records_raw)}")


Train records loaded: 1184
Test records loaded: 100


In [9]:
import re, json

# Same retrieval logic as notebook 1's KD teacher-prompting step, duplicated
# here since this is a separate Kaggle session. Used to build a per-report,
# ACR-augmented system prompt for both SFT training and inference (mod #3).
ACR_GUIDELINES_PATH = "/kaggle/input/datasets/mythreyeehari20/acr-rules-abdomen/Findings_Extracted_WhitePapers.json"

with open(ACR_GUIDELINES_PATH) as f:
    acr_guidelines = json.load(f)

def condense_finding(finding):
    feat_str = "; ".join(finding["features"])
    return f"[{finding['finding_id']}] {finding['finding_name']} \u2014 {feat_str}"

STOPWORDS = {"the","a","an","of","in","on","to","and","or","with","is","are","was","were",
             "at","for","by","as","be","no","not","also","this","that","been","has","have"}

def tokenize(text):
    words = re.findall(r"[a-z]+", text.lower())
    return set(
        w for w in words
        if w not in STOPWORDS and len(w) > 2
    )

def retrieve_relevant_findings(report_text, index, top_k=20, min_overlap=1):
    report_tokens = tokenize(report_text)
    scored = []
    for entry in index:
        finding_tokens = tokenize(entry["searchable_text"])
        overlap = len(report_tokens & finding_tokens)
        if overlap >= min_overlap:
            scored.append((overlap, entry))
    scored.sort(key=lambda x: -x[0])
    return [entry for _, entry in scored[:top_k]]

def build_filtered_acr_context(report_text, index, top_k=10):
    relevant = retrieve_relevant_findings(report_text, index, top_k=top_k)
    if not relevant:
        return full_condensed
    return "\n".join(e["condensed_line"] for e in relevant)

TASK_INSTRUCTIONS_TEMPLATE = (
    "You are a clinical assistant specialized in abdominopelvic radiology. Your task is to identify "
    "INCIDENTAL findings in a free-text abdominopelvic CT report — findings unrelated to the report's "
    "primary clinical indication, per ACR Incidental Findings Committee guidelines.\n\n"
    "Use the reference guidelines below to judge whether a finding is clinically incidental "
    "(e.g. small stable nodules, benign-appearing lymph nodes, calcifications) versus a primary/"
    "expected finding tied to the report's main indication.\n\n"
    "Extract the EXACT sentence(s) from the report that describe incidental findings — do not "
    "paraphrase or summarize. If no incidental findings are present, return an empty list.\n\n"
    "REFERENCE GUIDELINES:\n{acr_context}\n\n"
    "OUTPUT FORMAT:\n"
    "Return ONLY one JSON object and nothing else.\n"
    "The JSON object MUST contain exactly these two fields: "
    "contains_IF and incidental_sentences.\n"
    "contains_IF MUST be a boolean: true or false.\n"
    "incidental_sentences MUST be a JSON array of STRINGS, not objects.\n"
    "Each string must be an EXACT sentence copied from the report.\n"
    "Do NOT add fields such as sentence, if, is_incidental, findings, "
    "reference_guidelines, or any other fields.\n"
    "If there are no incidental findings, use an empty array and set contains_IF to false.\n"
    "If incidental findings are present, set contains_IF to true and include only "
    "the exact sentences containing those findings.\n"
    "Required format:\n"
    "{{\"contains_IF\": false, \"incidental_sentences\": []}}\n"
)

def build_condensed_index(guidelines):
    findings_list = guidelines["findings"]  # flat list, no organ_system nesting
    index = []
    for finding in findings_list:
        searchable = " ".join([finding["finding_name"], " ".join(finding["features"])]).lower()
        index.append({
            "finding_id": finding["finding_id"],
            "organ_system": "abdomen/pelvis",  # not present in this file, so hardcode or drop the field
            "searchable_text": searchable,
            "condensed_line": condense_finding(finding),
        })
    return index

ACR_INDEX = build_condensed_index(acr_guidelines)
full_condensed = "\n".join(f["condensed_line"] for f in ACR_INDEX)
print(f"Indexed {len(ACR_INDEX)} ACR findings ({len(full_condensed):,} chars condensed)")

Indexed 27 ACR findings (13,332 chars condensed)


In [10]:
def build_chat_messages(report_text, acr_index, top_k=20):
    filtered_context = build_filtered_acr_context(report_text, acr_index, top_k=top_k)
    instructions = TASK_INSTRUCTIONS_TEMPLATE.format(acr_context=filtered_context)
    messages = [{"role": "system", "content": instructions}]
    for demo in fewshot_raw:
        messages.append({"role": "user", "content": f"Report:\n{demo['messages']}"})
        messages.append({"role": "assistant", "content": json.dumps(demo["gold"])})
    messages.append({"role": "user", "content": f"Report:\n{report_text}"})
    return messages

def build_system_prompt(report_text, top_k=20):
    filtered_context = build_filtered_acr_context(report_text, ACR_INDEX, top_k=top_k)
    return TASK_INSTRUCTIONS_TEMPLATE.format(acr_context=filtered_context)

In [11]:
eval_model = AutoModelForCausalLM.from_pretrained(
    FINAL_MERGED_DIR, torch_dtype=torch.bfloat16, device_map={"": 0}, trust_remote_code=True,
)
eval_model.eval()

sanity_results = generate_predictions(eval_model, tokenizer, test_records_raw, n=None)
sanity_metrics = run_evaluation_fuzzy(sanity_results, label="Merged KD+QLoRA (sanity check)")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


=== Merged KD+QLoRA (sanity check) (fuzzy threshold=0.85) ===
Parse failures:      1/100
Negative-report acc: 0.9508  (n=61)
Positive-report F1:  0.4737  (n=38)
Sentence Macro F1:   0.7123
Sentence Weighted:   0.7677
Sentence Micro F1:   0.5250


In [12]:
from tqdm import tqdm

test_structured_records = test_records_raw  # ← moved up here, before first use

def generate_sampled(model, tokenizer, record, temperature, max_new_tokens=256):
    messages = build_messages(record)
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

TEMPERATURES = [0.5, 0.7, 0.9]
N_SAMPLES = 5

all_temp_predictions = {}
for temp in TEMPERATURES:
    self_consistency_predictions = {}
    for i in range(N_SAMPLES):
        results = []
        for rec in tqdm(test_structured_records, desc=f"T={temp} sample {i}"):
            raw_output = generate_sampled(eval_model, tokenizer, rec, temperature=temp)
            parsed = parse_output(raw_output)
            results.append({
                "pred_sentences": parsed.get("incidental_sentences") if parsed else None,
                "parse_failed": parsed is None,
            })
        self_consistency_predictions[f"sample_{i}"] = results
    all_temp_predictions[temp] = self_consistency_predictions
    with open(f"final_self_consistency_T{temp}.json", "w") as f:
        json.dump(self_consistency_predictions, f, indent=2)

T=0.9 sample 4: 100%|██████████| 100/100 [03:30<00:00,  2.11s/it]


In [13]:
test_structured_records = test_records_raw

def combine_predictions_threshold(predictions_dict, min_agree_fraction, records):
    member_names = list(predictions_dict.keys())
    combined = []
    for i, record in enumerate(records):
        gold_sentences = extract_gold(record)["incidental_sentences"]
        member_sets = [set(s.strip().lower() for s in (predictions_dict[n][i]["pred_sentences"] or []))
                       for n in member_names if not predictions_dict[n][i]["parse_failed"]]
        if not member_sets:
            combined.append({"gold_sentences": gold_sentences, "pred_sentences": [], "parse_failed": True})
            continue
        all_sentences = set().union(*member_sets)
        threshold = min_agree_fraction * len(member_sets)
        final = {s for s in all_sentences if sum(s in ms for ms in member_sets) >= threshold}
        combined.append({"gold_sentences": gold_sentences, "pred_sentences": list(final), "parse_failed": False})
    return combined

full_sweep_results = {}
for temp, preds in all_temp_predictions.items():
    for frac in [0.3, 0.4, 0.5, 0.6, 0.7]:
        combined = combine_predictions_threshold(preds, frac, test_structured_records)
        m = run_evaluation_fuzzy(combined, label=f"T={temp}, Threshold={frac}", threshold=0.85)
        full_sweep_results[(temp, frac)] = m

best_config = max(full_sweep_results.items(), key=lambda x: x[1]["macro_f1"])
print(f"Best config: T={best_config[0][0]}, Threshold={best_config[0][1]}")
print(f"Macro F1: {best_config[1]['macro_f1']:.4f} (sanity-check greedy baseline was {sanity_metrics['macro_f1']:.4f})")


=== T=0.5, Threshold=0.3 (fuzzy threshold=0.85) ===
Parse failures:      0/100
Negative-report acc: 0.9508  (n=61)
Positive-report F1:  0.5275  (n=39)
Sentence Macro F1:   0.7391
Sentence Weighted:   0.7857
Sentence Micro F1:   0.5591

=== T=0.5, Threshold=0.4 (fuzzy threshold=0.85) ===
Parse failures:      0/100
Negative-report acc: 0.9508  (n=61)
Positive-report F1:  0.5275  (n=39)
Sentence Macro F1:   0.7391
Sentence Weighted:   0.7857
Sentence Micro F1:   0.5591

=== T=0.5, Threshold=0.5 (fuzzy threshold=0.85) ===
Parse failures:      0/100
Negative-report acc: 0.9508  (n=61)
Positive-report F1:  0.4249  (n=39)
Sentence Macro F1:   0.6879
Sentence Weighted:   0.7457
Sentence Micro F1:   0.4884

=== T=0.5, Threshold=0.6 (fuzzy threshold=0.85) ===
Parse failures:      0/100
Negative-report acc: 0.9508  (n=61)
Positive-report F1:  0.4249  (n=39)
Sentence Macro F1:   0.6879
Sentence Weighted:   0.7457
Sentence Micro F1:   0.4884

=== T=0.5, Threshold=0.7 (fuzzy threshold=0.85) ===
Par